In [33]:
import mne 
import torch
import numpy as np
import os 
from sklearn.model_selection import train_test_split

In [ ]:
np.random.seed(42)  # 再現性（ダウンサンプリングの乱数固定)

In [34]:
subject_ids = range(1,110)
all_x,all_y,all_groups = [],[],[]

In [35]:
base_path = "E:/EEG_Dataset/eeg-motor-movementimagery-dataset-1.0.0"

In [36]:
# 左右MIが入ってるラン：4 / 8 / 12
RUNS  = ["R04", "R08", "R12"]

# （チャンネルはそのままでもOKだけど、MI寄りには↓が定番）
# PICKS = ["FC3","C3","CP3","FCz","Cz","CPz","FC4","C4"]
# PICKS = ["FC1", "FC2", "C3", "CZ", "C4", "CP1", "CP2", "PZ"]  # ←今使ってるやつ
PICKS = ["FC1", "FC2", "C3", "C4", "CP1", "CP2", "P3", "P4"]


all_x, all_y, all_groups = [], [], []

for subject_id in subject_ids:
    subj = f"S{subject_id:03d}"
    for run in RUNS:
        edf_path = os.path.join(base_path, subj, f"{subj}{run}.edf")
        if not os.path.exists(edf_path):
            continue

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")
        raw.rename_channels(lambda s: s.replace('.','').upper())
        raw.set_montage('standard_1020', on_missing='ignore')
        raw, _ = mne.set_eeg_reference(raw, ref_channels='average')
        # 事前チェック（無いchがあればこのrunはスキップ）
        missing = [ch for ch in PICKS if ch not in raw.ch_names]
        if missing:
            print(f"[WARN] {subj}{run}: missing channels {missing} → skip")
            continue
        raw.pick_channels(PICKS, ordered=True)
        assert raw.ch_names == PICKS, f"channel order mismatch: {raw.ch_names} != {PICKS}"

        raw.filter(1., 30., fir_design='firwin')
        raw.resample(250)

        events, _ = mne.events_from_annotations(raw)
        event_id = {'T1': 1, 'T2': 2}  # ← このラン群は T1=左, T2=右 

        epochs = mne.Epochs(raw, events, event_id=event_id,
                            tmin=-0.5, tmax=3.5, baseline=(-0.5,0.0),
                            preload=True, verbose="ERROR")

        x = epochs.get_data()
        y_ids = epochs.events[:, 2]
        y = np.where(y_ids == event_id['T1'], 0, 1)  # 左→0, 右→1

        all_x.append(x)
        all_y.append(y)
        all_groups.append(np.full(len(y), subject_id, dtype=int))  # グループ=被験者ID


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 529 samples (3.306 s)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_channels() is a legacy function. New code should u

In [37]:
all_x_fixed = []
all_y_fixed = []
for x, y in zip(all_x, all_y):
    if x.shape[2] == 1001:
        all_x_fixed.append(x)
        all_y_fixed.append(y)

x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)
groups = np.concatenate(all_groups, axis=0)

print("all:", x_all.shape, y_all.shape)  # 例：(試行数, 8チャネル, 時間点数)

all: (7070, 8, 1001) (7070,)


In [46]:
fs = 250
tmin, tmax = -0.5, 3.5
expected_T = int(round((tmax - tmin) * fs)) + 1  # = 1001（MNEは両端含む）

all_x_fixed, all_y_fixed, all_groups = [], [], []
for file_idx, (x, y) in enumerate(zip(all_x, all_y)):
    T = x.shape[2]
    if T >= expected_T:
        # 長い分は末尾を切り落として1001に合わせる（≒オフバイワン修正）
        x = x[..., :expected_T]
    else:
         #まれに短い場合だけスキップ or 末尾パディング（必要なら）
        #pad = expected_T - T
        #x = np.pad(x, ((0,0),(0,0),(0,pad)), mode='edge')
        pass
    all_x_fixed.append(x)
    all_y_fixed.append(y)
    all_groups.append(np.full(len(y), file_idx, dtype=int))
x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)
groups = np.concatenate(all_groups, axis=0)
print("all:", x_all.shape, y_all.shape)

all: (7070, 8, 1001) (7070,)


In [38]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(x_all, y_all, groups))

X_train, X_test = x_all[train_idx], x_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
groups_train = groups[train_idx]

# リーク防止チェック
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
#print("train/test:", X_train.shape, X_test.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

In [39]:
#学習データのクラスバランス調整
idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]
min_len = min(len(idx0), len(idx1))
idx0_ds = np.random.choice(idx0, min_len, replace=False)
idx1_ds = np.random.choice(idx1, min_len, replace=False)
balanced_idx = np.concatenate([idx0_ds, idx1_ds])

X_train = X_train[balanced_idx]
y_train = y_train[balanced_idx]

In [ ]:
# 正規化
X_train = (X_train - X_train.mean(axis=2, keepdims=True)) / X_train.std(axis=2, keepdims=True)
X_test  = (X_test  - X_test.mean(axis=2, keepdims=True))  / X_test.std(axis=2, keepdims=True)

In [40]:
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, C, T)
        self.y = torch.tensor(y, dtype=torch.long)     # (N,)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [41]:
from torch.utils.data import DataLoader

train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [42]:
import torch.nn as nn
import torch.nn.functional as F

class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=16, samples=641):
        super(EEGNet, self).__init__()

        # 第1層: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        # 第2層: Depthwise Convolution (Spatial filter)
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(p=0.25)
        )

        # 第3層: Separable Convolution
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(p=0.25)
        )

        # 全結合層
        self.classify = nn.Linear(self._get_flattened_size(channels, samples), num_classes)

    def _get_flattened_size(self, channels, samples):
        x = torch.zeros(1, 1, channels, samples)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        return x.numel()

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, C, T)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        x = x.view(x.size(0), -1)
        return self.classify(x)

In [ ]:
device = torch.device("cuda")  

model = EEGNet(num_classes=2, channels=8, samples=1001).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)


In [44]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    acc = correct / total
    return running_loss / len(loader), acc

def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    return correct / total

In [45]:
epochs = 50  # 時間と相談して増減してOK

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

Epoch 1/50 - Train Loss: 0.6381, Train Acc: 0.6561, Test Acc: 0.6489
Epoch 2/50 - Train Loss: 0.5762, Train Acc: 0.7130, Test Acc: 0.6517
Epoch 3/50 - Train Loss: 0.5519, Train Acc: 0.7340, Test Acc: 0.7093
Epoch 4/50 - Train Loss: 0.5268, Train Acc: 0.7433, Test Acc: 0.7303
Epoch 5/50 - Train Loss: 0.5272, Train Acc: 0.7425, Test Acc: 0.5941
Epoch 6/50 - Train Loss: 0.5062, Train Acc: 0.7607, Test Acc: 0.7598
Epoch 7/50 - Train Loss: 0.5029, Train Acc: 0.7595, Test Acc: 0.7395
Epoch 8/50 - Train Loss: 0.5002, Train Acc: 0.7658, Test Acc: 0.7395
Epoch 9/50 - Train Loss: 0.4820, Train Acc: 0.7686, Test Acc: 0.7374
Epoch 10/50 - Train Loss: 0.4603, Train Acc: 0.7888, Test Acc: 0.7591
Epoch 11/50 - Train Loss: 0.4688, Train Acc: 0.7835, Test Acc: 0.6657
Epoch 12/50 - Train Loss: 0.4681, Train Acc: 0.7784, Test Acc: 0.7367
Epoch 13/50 - Train Loss: 0.4674, Train Acc: 0.7787, Test Acc: 0.7437
Epoch 14/50 - Train Loss: 0.4594, Train Acc: 0.7772, Test Acc: 0.7683
Epoch 15/50 - Train Loss: 0.4

In [ ]:
# モデルの重み保存
torch.save(model.state_dict(), 'eegnet_model.pt')

In [ ]:
np.bincount(y_train)